# 🤖 全自動 A100 遠端算圖 & 算影片雲端節點 (`auto_worker.ipynb`)

這份筆記本由本機 **Antigravity Agent** 自動控制與監聽：
1. **自動讀取任務與動態隨機 Prompt**：鎖定「20歲法越混血、白皙透亮膚質、165cm/50kg 自然 F 罩杯沙漏體態」核心人設，每張圖自動隨機變換**臉型、髮色、髮型、妝容、姿勢、服裝款式/顏色與背景**！
2. **即時回傳成品**：每算完一張照片（`.jpg`）或一支影片（`.mp4`），自動透過加密通道將檔案直接傳回你本機電腦的 `outputs/` 資料夾！
3. **自動切斷 A100 省額度**：本機確認收到所有照片與影片後，自動呼叫 `runtime.unassign()` **立刻中斷並刪除 A100 執行階段**！

In [ ]:
# ==================== 全自動雲端執行引擎 (單一區塊一鍵跑完) ====================
import os, sys, json, time, shutil, subprocess, threading, re, gc
from http.server import SimpleHTTPRequestHandler, HTTPServer
import urllib.request

print("🚀 [1/6] 正在安裝必要套件與 Cloudflare Tunnel...")
subprocess.run("pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg ftfy", shell=True, check=True)
if not os.path.exists("/usr/local/bin/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared", shell=True, check=True)

print("🔄 [2/6] 正在從 GitHub 同步最新任務設定與隨機 Prompt 引擎...")
if not os.path.exists("colab-ai-media-lab"):
    subprocess.run("git clone https://github.com/Chenponpon/colab-ai-media-lab.git", shell=True, check=True)
else:
    subprocess.run("cd colab-ai-media-lab && git fetch --all && git reset --hard origin/main", shell=True, check=True)

sys.path.insert(0, os.path.abspath("colab-ai-media-lab/scripts"))
from prompt_randomizer import generate_diversified_prompt

with open("colab-ai-media-lab/queue/current_job.json", "r", encoding="utf-8") as f:
    job = json.load(f)

NTFY_TOPIC = job.get("ntfy_topic", "chenponpon_agy_colab_9f8a7b6c5d")
JOB_ID = job.get("job_id", "default_job")
OUT_DIR = os.path.abspath("outputs")
os.makedirs(OUT_DIR, exist_ok=True)
download_ack = threading.Event()

def send_event(event_type, message, extra=None):
    payload = {"job_id": JOB_ID, "event": event_type, "message": message, "timestamp": time.time()}
    if extra:
        payload.update(extra)
    data = json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(f"https://ntfy.sh/{NTFY_TOPIC}", data=data, method="POST")
    try:
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        print(f"⚠️ 通知發送失敗: {e}")

class WorkerHandler(SimpleHTTPRequestHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, directory=OUT_DIR, **kwargs)
    def do_POST(self):
        if self.path == "/ack":
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b"OK")
            download_ack.set()
    def log_message(self, format, *args):
        pass

httpd = HTTPServer(("0.0.0.0", 8765), WorkerHandler)
threading.Thread(target=httpd.serve_forever, daemon=True).start()

cf_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://localhost:8765"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
tunnel_url = None
for _ in range(60):
    line = cf_proc.stdout.readline()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        tunnel_url = m.group(0)
        break

import torch
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3) if torch.cuda.is_available() else 0
print(f"✅ 遠端傳輸通道已建立: {tunnel_url} | GPU: {gpu_name} ({vram_gb:.1f} GB)")
send_event("WORKER_ONLINE", f"Colab 已連線: {gpu_name} ({vram_gb:.1f} GB)", {"tunnel_url": tunnel_url})

from PIL import Image
generated_image_paths = []

# ==================== 階段一：自動多樣化生圖 ====================
img_cfg = job.get("image_generation", {})
if img_cfg.get("enabled", True):
    print("🎨 [3/6] 正在執行寫實人像多樣化生圖任務...")
    send_event("STATUS", "正在載入 RealVisXL V5.0 寫實人像模型...")
    from diffusers import AutoPipelineForText2Image, AutoPipelineForImage2Image, DPMSolverMultistepScheduler
    model_id = img_cfg.get("model_id", "SG161222/RealVisXL_V5.0")
    use_base = img_cfg.get("use_ref_image_as_base", False)
    if use_base:
        pipe = AutoPipelineForImage2Image.from_pretrained(model_id, torch_dtype=torch.float16, variant="fp16").to("cuda")
    else:
        pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")

    w, h = img_cfg.get("width", 832), img_cfg.get("height", 1216)
    randomize = img_cfg.get("randomize_prompt", True)

    for i in range(img_cfg.get("num_images", 4)):
        t0 = time.time()
        if randomize:
            active_prompt, recipe = generate_diversified_prompt()
            recipe_file = f"generated_portrait_{i+1}_recipe.json"
            with open(os.path.join(OUT_DIR, recipe_file), "w", encoding="utf-8") as rf:
                json.dump(recipe, rf, ensure_ascii=False, indent=2)
            print(f"🎲 第 {i+1} 張隨機場景: {recipe['background'][:60]}...")
        else:
            active_prompt = img_cfg["prompt"]

        if use_base:
            base_p = os.path.join("colab-ai-media-lab", img_cfg.get("base_image", "assets/ref_1.jpg"))
            init_img = Image.open(base_p).convert("RGB").resize((w, h), Image.LANCZOS)
            img = pipe(
                prompt=active_prompt, negative_prompt=img_cfg.get("negative_prompt", ""),
                image=init_img, strength=img_cfg.get("denoising_strength", 0.65),
                num_inference_steps=img_cfg.get("steps", 30), guidance_scale=img_cfg.get("guidance_scale", 5.5)
            ).images[0]
        else:
            img = pipe(
                prompt=active_prompt, negative_prompt=img_cfg.get("negative_prompt", ""),
                width=w, height=h,
                num_inference_steps=img_cfg.get("steps", 30), guidance_scale=img_cfg.get("guidance_scale", 5.5)
            ).images[0]
        fname = f"generated_portrait_{i+1}.jpg"
        fpath = os.path.join(OUT_DIR, fname)
        img.save(fpath, quality=95)
        generated_image_paths.append(fpath)
        dt = time.time() - t0
        print(f"✅ 生圖完成: {fname} ({dt:.1f}s)")
        send_event("FILE_READY", f"生圖完成: {fname} ({dt:.1f}s)", {"filename": fname, "url": f"{tunnel_url}/{fname}"})
        if randomize:
            send_event("FILE_READY", f"隨機配方紀錄: {recipe_file}", {"filename": recipe_file, "url": f"{tunnel_url}/{recipe_file}"})
    del pipe
    gc.collect()
    torch.cuda.empty_cache()

# ==================== 階段二：自動圖生影片 (Wan 2.1 I2V 14B) ====================
vid_cfg = job.get("video_generation", {})
if vid_cfg.get("enabled", True):
    print("🎬 [4/6] 正在載入 Wan 2.1 I2V 14B 旗艦圖生影模型...")
    send_event("STATUS", "正在載入 Wan 2.1 I2V 14B 旗艦圖生影片模型 (首次下載權重約需 2~3 分鐘)...")
    from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
    from diffusers.utils import export_to_video
    from transformers import CLIPVisionModel

    v_model = vid_cfg.get("model_id", "Wan-AI/Wan2.1-I2V-14B-480P-Diffusers")
    image_encoder = CLIPVisionModel.from_pretrained(v_model, subfolder="image_encoder", torch_dtype=torch.float32)
    vae = AutoencoderKLWan.from_pretrained(v_model, subfolder="vae", torch_dtype=torch.float32)
    v_pipe = WanImageToVideoPipeline.from_pretrained(v_model, vae=vae, image_encoder=image_encoder, torch_dtype=torch.bfloat16)
    v_pipe.enable_model_cpu_offload()
    v_pipe.vae.enable_tiling()

    targets = [os.path.join("colab-ai-media-lab", p) for p in vid_cfg.get("input_images", [])]
    if vid_cfg.get("animate_generated_images", False):
        max_gen = vid_cfg.get("max_generated_to_animate", 1)
        targets.extend(generated_image_paths[:max_gen])

    vw, vh = vid_cfg.get("width", 544), vid_cfg.get("height", 832)
    for idx, img_p in enumerate(targets):
        t0 = time.time()
        send_event("STATUS", f"正在算第 {idx+1}/{len(targets)} 支動態影片 ({os.path.basename(img_p)})...")
        in_img = Image.open(img_p).convert("RGB").resize((vw, vh), Image.LANCZOS)
        frames = v_pipe(
            image=in_img,
            prompt=vid_cfg["motion_prompt"],
            negative_prompt=vid_cfg.get("negative_prompt", ""),
            height=vh, width=vw,
            num_frames=vid_cfg.get("num_frames", 81),
            guidance_scale=vid_cfg.get("guidance_scale", 5.0),
            num_inference_steps=vid_cfg.get("steps", 25)
        ).frames[0]
        vname = f"animated_video_{idx+1}.mp4"
        vpath = os.path.join(OUT_DIR, vname)
        export_to_video(frames, vpath, fps=vid_cfg.get("fps", 16))
        dt = time.time() - t0
        print(f"✅ 影片完成: {vname} ({dt:.1f}s)")
        send_event("FILE_READY", f"影片完成: {vname} ({dt:.1f}s)", {"filename": vname, "url": f"{tunnel_url}/{vname}"})

print("📦 [5/6] 所有圖片與影片已生成完畢！等待本機電腦確認下載完成...")
send_event("JOB_COMPLETED", "所有圖片與影片皆已生成！等待本機確認接收後自動關閉 A100...", {"tunnel_url": tunnel_url})

download_ack.wait(timeout=180)
time.sleep(3)

if job.get("auto_shutdown_a100", True):
    print("🛑 [6/6] 本機已接收所有檔案，正在自動中斷並釋放 A100 執行階段以節省額度！")
    send_event("SHUTDOWN", "A100 執行階段已自動中斷並釋放，不消耗額外配額！")
    from google.colab import runtime
    runtime.unassign()